# Геолокация V4 · 50 новых вариантов

Используем прогнозы V3 для 28 способов объединения вероятностей и локальной плотности и 22 обучаемых способов взвешивания и коррекции. Сравниваем все 50 с прежним победителем V3. Критерий выбора — RMSE геодезических расстояний в километрах по WGS84, дополнительная метрика — MAE.

![Схема V4](docs/workflow-v4.svg)

## 1. Среда и данные

Выберите GPU Colab и используйте тот же `GEO_CV_ROOT`, что в [V3](train250k_v3.ipynb). В новом сеансе загрузите актуальный ZIP проекта и укажите `ZIP_PATH`. Если проект уже распакован, укажите путь `REPO_DIR`.

V4 использует сохранённые внешние, внутренние и финальные прогнозы V3. Ноутбук восстановит необходимые этапы V3 при отсутствии результатов. Новые артефакты сохраняются в подкаталоге `v4`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from IPython.display import SVG, display

REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
TRAIN_250K_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
GEO_CV_ROOT = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
here = Path.cwd()
repo = next((p for p in (checkout, here, here.parent) if (p / "src" / "run_v4.py").is_file()), None)
if repo is None:
    if not ZIP_PATH:
        raise FileNotFoundError("Укажите ZIP_PATH или путь к новой версии проекта в REPO_DIR.")
    if checkout.exists():
        raise FileExistsError("Для нового ZIP укажите свободный путь в REPO_DIR.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        roots = [Path(tmp), *Path(tmp).iterdir()]
        source = next((p for p in roots if (p / "src" / "run_v4.py").is_file()), None)
        if source is None:
            raise FileNotFoundError("В ZIP отсутствует src/run_v4.py.")
        from shutil import copytree
        copytree(source, checkout)
    repo = checkout

data, work = (Path(p).expanduser().resolve() for p in (TRAIN_250K_DIR, GEO_CV_ROOT))
env = {**os.environ, "TRAIN_250K_DIR": str(data), "GEO_CV_ROOT": str(work)}
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите среду Colab с GPU.")
print(torch.cuda.get_device_name())

def run_src(script, *args):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *args], cwd=repo, env=env, check=True)

if (repo / "docs" / "workflow-v4.svg").is_file():
    display(SVG(filename=str(repo / "docs" / "workflow-v4.svg")))

In [ ]:
if not (work / "split.json").is_file():
    if (data.is_relative_to("/content/drive") or work.is_relative_to("/content/drive")) and not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    run_src("prepare.py")
split = json.loads((work / "split.json").read_text())
assert split["trainval"] == 212_500 and split["test"] == 37_500
assert split["fold_sizes"] == [42_500] * 5
print("Разбиение подтверждено", split["fold_sizes"], "тест", split["test"])

## 2. Сравнение 50 новых вариантов

| Семейство | Количество | Что проверяем |
|---|---:|---|
| Вероятности ячеек | 16 | Температура, робастное усреднение, географический консенсус, выбор масштаба |
| Географическая плотность | 12 | Перекрёстная поддержка ячеек, уточнение соседями, ковариация, априорные частоты |
| Обучаемые стеки | 12 | Остатки, риск экспертов, локальная надёжность, глобальные и региональные смеси |
| Калибровка WGS84 | 10 | Геодезические смещения, условные веса, пространственная коррекция, признаки PCA |

Каждому внешнему фолду соответствуют свои прогнозы V3. Обучаемые методы получают OOF-признаки из трёх внутренних разбиений внешних 170 000 снимков. Окончательную оценку фолда считаем по 42 500 внешним снимкам. В лидерборде 51 вариант с учётом зафиксированного победителя V3.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

v3 = work / "v3"
cv_files = ["split.sha256", "cv-summary.json"]
cv_files += [f"bundle-o{fold}.npz" for fold in range(5)]
cv_files += [f"bundle-o{fold}-i{i}.npz" for fold in range(5) for i in range(3)]
if any(not (v3 / filename).is_file() for filename in cv_files):
    run_src("run_v3.py", "cv")
missing = [filename for filename in cv_files if not (v3 / filename).is_file()]
if missing:
    raise FileNotFoundError(f"Неполные артефакты V3: {missing}. Укажите GEO_CV_ROOT с полной копией или восстановите V3 в новом каталоге.")

run_src("run_v4.py", "cv")
cv = json.loads((work / "v4" / "cv-summary.json").read_text())
ranking = pd.DataFrame(cv["candidates"]).sort_values("cv_rmse_km", ignore_index=True)
assert cv["n_new_variants"] == 50 and len(ranking) == 51
assert ranking["name"].nunique() == 51 and ranking.iloc[0]["name"] == cv["winner"]
incumbent = ranking.loc[ranking["name"] == "incumbent_v3"].iloc[0]
ranking["delta_rmse_km"] = ranking["cv_rmse_km"] - incumbent["cv_rmse_km"]
ranking.index += 1
print(f"Победитель по пяти фолдам: {cv['winner']}")

columns = ["name", "family", "kind", "description", "cv_mae_km", "cv_rmse_km", "delta_rmse_km"]
table = ranking[columns].rename(columns={
    "name": "Вариант", "family": "Семейство", "kind": "Тип",
    "description": "Идея", "cv_mae_km": "MAE, км",
    "cv_rmse_km": "RMSE, км", "delta_rmse_km": "Δ RMSE к V3, км",
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(table.round(2))
folds = pd.DataFrame(ranking["fold_rmse_km"].tolist(), index=ranking["name"],
                     columns=[f"Фолд {i + 1}" for i in range(5)])
with pd.option_context("display.max_rows", None):
    display(folds.round(2))

best = ranking.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(best["cv_rmse_km"], best["name"], color="#1e749b", s=45, zorder=3)
ax.axvline(incumbent["cv_rmse_km"], color="#a16207", linestyle="--", label="V3")
ax.set_xlabel("RMSE на валидации, км")
ax.set_title("Первые 12 вариантов по RMSE")
ax.grid(axis="x", alpha=.2)
ax.legend()
fig.tight_layout()
plt.show()

## 3. Итоговый тест

После фиксации победителя по CV используем финальный пакет V3, обученный на 212 500 снимках разработки. Обучаемый способ дополнительно переобучается на OOF-прогнозах всех пяти внешних частей. Если победителем остаётся V3, берём его сохранённые test-прогнозы и метрики.

Тест из 37 500 снимков неизменен. Его прежние агрегаты служат ориентиром, гиперпараметры V4 выбираются только по валидационной части.

In [ ]:
final_files = ["bundle-final.npz", "results.json", "test-predictions.npz"]
if any(not (v3 / filename).is_file() for filename in final_files):
    # V3 уже выбрал свою конфигурацию в собственном CV.
    run_src("run_v3.py", "final")
if any(not (v3 / filename).is_file() for filename in final_files):
    raise FileNotFoundError("Восстановите полный финальный пакет V3 или укажите каталог с ним.")

run_src("run_v4.py", "final")
result = json.loads((work / "v4" / "results.json").read_text())
assert result["winner"] == cv["winner"]
test = result["test"]
assert test["n"] == 37_500
previous = result["historical_reference"]
comparison = [
    {"Модель": "V3 · исторический результат", "MAE, км": previous["mae_km"],
     "RMSE, км": previous["rmse_km"]},
    {"Модель": result["winner"], "MAE, км": test["mae_km"], "RMSE, км": test["rmse_km"]},
]
display(pd.DataFrame(comparison).round(2))
delta = test["rmse_km"] - previous["rmse_km"]
print(f"Δ RMSE к V3: {delta:+.2f} км ({100 * delta / previous['rmse_km']:+.2f}%)")